In [1]:
import torch, platform, sys
print("torch:", torch.__version__)
print("cuda_is_available:", torch.cuda.is_available(), "device_count:", torch.cuda.device_count())
print("torch.version.cuda:", torch.version.cuda)


torch: 2.5.1+cu121
cuda_is_available: True device_count: 2
torch.version.cuda: 12.1


In [ ]:
"""
Calculating metrics for each image and model
"""
import os, re, pickle, torch
import torch.nn.functional as F
import pandas as pd

DTYPE = torch.float32        
EPS = 1e-8
_TEMP_RE = re.compile(r"(?:^|[_-])temp[_-]?([0-9]+(?:\.[0-9]+)?)")
def parse_temp(fname: str):
    m = _TEMP_RE.search(fname)
    return float(m.group(1)) if m else None

def load_logits(path: str) -> torch.Tensor:
    with open(path, "rb") as f:
        d = pickle.load(f)
    x = d["logits"]
    if isinstance(x, torch.Tensor):
        return x.to(dtype=DTYPE, device="cpu", copy=False)
    else:
        # olası numpy -> torch
        return torch.as_tensor(x, dtype=DTYPE)

def align_minlen(a: torch.Tensor, b: torch.Tensor):
    # [B,T,V] bekleniyor
    T = min(a.size(1), b.size(1))
    return a[:, :T, :], b[:, :T, :]

@torch.no_grad()
def precompute_for_file(tensor_logits: torch.Tensor):

    B, T, V = tensor_logits.shape
    logPa = F.log_softmax(tensor_logits, dim=-1)
    Pa    = logPa.exp()
    # entropy H(P) = -sum Pa*logPa
    H = (-Pa * logPa).sum(dim=-1)  # [B,T]
    flat = tensor_logits.reshape(B, T * V)
    flat_norm = flat.norm(p=2, dim=-1, keepdim=True).clamp_min(EPS)
    flat_normed = flat / flat_norm
    return {
        "logits": tensor_logits,        # [B,T,V]
        "Pa": Pa,                       # [B,T,V]
        "logPa": logPa,                 # [B,T,V]
        "H": H,                         # [B,T]
        "flat_normed": flat_normed,     # [B, T*V]
    }

@torch.no_grad()
def pair_metrics(pre_i, pre_j):

    Ai, Aj = pre_i["logits"], pre_j["logits"]
    Ai, Aj = align_minlen(Ai, Aj)             # [B,T,V]
    B, T, V = Ai.shape

    fi = pre_i["flat_normed"][:, :T*V]
    fj = pre_j["flat_normed"][:, :T*V]
    cosine_flat = torch.sum(fi * fj, dim=-1).mean()

    # KL : KL(P||Q) = sum P * (logP - logQ)
    Pai, logPai = pre_i["Pa"][:, :T, :], pre_i["logPa"][:, :T, :]
    Paj, logPaj = pre_j["Pa"][:, :T, :], pre_j["logPa"][:, :T, :]
    kl_ij_tok = (Pai * (logPai - logPaj)).sum(dim=-1)
    kl_ji_tok = (Paj * (logPaj - logPai)).sum(dim=-1)
    kl_ab = kl_ij_tok.mean()
    kl_ba = kl_ji_tok.mean()

    # JS(P||Q) = H(M) - (H(P)+H(Q))/2, M=(P+Q)/2
    Hi = pre_i["H"][:, :T]        # [B,T]
    Hj = pre_j["H"][:, :T]
    M = 0.5 * (Pai + Paj)         # [B,T,V]
    logM = (M.clamp_min(EPS)).log()
    Hm = (-M * logM).sum(dim=-1)  # [B,T]
    js = (Hm - 0.5 * (Hi + Hj)).mean().clamp_min(0.0)

    mae = F.l1_loss(Ai, Aj)

    return (
        float(cosine_flat.item()),
        float(kl_ab.item()),
        float(kl_ba.item()),
        float(js.item()),
        float(mae.item()),
        int(T),
    )

def run_one_image_folder_fast(pkl_folder: str, out_xlsx_path: str, target_temps):
    files = [f for f in os.listdir(pkl_folder) if f.lower().endswith(".pkl")]
    if not files:
        raise FileNotFoundError(f"No pkl at {pkl_folder}")
    print("[scan]", pkl_folder)

    by_temp = {t: [] for t in target_temps}
    for f in files:
        t = parse_temp(f)
        if t is not None and any(abs(t - tt) < 1e-6 for tt in target_temps):
            by_temp[min(target_temps, key=lambda x: abs(x - t))].append(f)

    with pd.ExcelWriter(out_xlsx_path, engine="xlsxwriter") as writer:
        for t in target_temps:
            group = sorted(by_temp.get(t, []))
            if not group:
                continue
            cache = []
            for fname in group:
                logits = load_logits(os.path.join(pkl_folder, fname))
                if logits.dim() == 2:
                    logits = logits.unsqueeze(0)
                cache.append({
                    "name": fname,
                    "pre": precompute_for_file(logits)
                })

            results = []
            N = len(cache)
            for i in range(N):
                for j in range(i, N):
                    ci, cj = cache[i], cache[j]
                    cos, kl_ab, kl_ba, js, mae, Tused = pair_metrics(ci["pre"], cj["pre"])
                    results.append({
                        "Base": i+1, "Other": j+1, "Temperature": t,
                        "file_i": ci["name"], "file_j": cj["name"],
                        "cosine_flat": cos, "kl_ab": kl_ab, "kl_ba": kl_ba,
                        "js_divergence": js, "mae": mae, "used_seq_len": Tused
                    })
                    if i != j:
                        results.append({
                            "Base": j+1, "Other": i+1, "Temperature": t,
                            "file_i": cj["name"], "file_j": ci["name"],
                            "cosine_flat": cos, "kl_ab": kl_ba, "kl_ba": kl_ab,
                            "js_divergence": js, "mae": mae, "used_seq_len": Tused
                        })

            df = pd.DataFrame(results)
            sheet_name = f"temp_{t}".replace(".", "_")
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"[OK] Excel: {out_xlsx_path}")
    
if __name__ == "__main__": 

    for model_name in  [ "VILA8B"]: #["VILA8B", "LLAVAMed", "PRISM"] 
        for ques_no in [2,3]: #1,2,3 
            for img_num in range(25): # number of images to analyze
                img_num = img_num + 126 # from which img to start
                pkl_folder = rf"D:\HPdata\exp1_pkl_files\pkl_outputs_{model_name}_q{ques_no}\pkl_output_{img_num}" 
                out_xlsx = rf"D:\HPdata\exp1_pkl_files\pkl_outputs_{model_name}_q{ques_no}\metrics_q{ques_no}_img{img_num}.xlsx" 
                target_temps = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0] 
                run_one_image_folder_fast(pkl_folder, out_xlsx, target_temps=target_temps)


[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_126
[OK] Excel: D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\metrics_q2_img126.xlsx
[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_127
[OK] Excel: D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\metrics_q2_img127.xlsx
[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_128
[OK] Excel: D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\metrics_q2_img128.xlsx
[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_129
[OK] Excel: D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\metrics_q2_img129.xlsx
[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_130
[OK] Excel: D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\metrics_q2_img130.xlsx
[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_131
[OK] Excel: D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\metrics_q2_img131.xlsx
[scan] D:\HPdata\exp1_pkl_files\pkl_outputs_VILA8B_q2\pkl_output_132
[OK] Excel: D:\HPdata\exp

In [ ]:
"""
Parsings, normalizing, and visualizing evaluation metric data from Excel files 
across multiple models, questions, and temperatures, saving both summary spreadsheets 
and comparative trend plots.
"""
import os, re, glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

MODELS   = ["VILA8B", "LLAVAMed", "PRISM"]
QUESTIONS = [1, 2, 3]
BASE_DIR = r"D:\HPdata\exp1_pkl_files" 
OUT_DIR  = os.path.join(BASE_DIR, "plots_combined_normalized_with_std_fixed")
os.makedirs(OUT_DIR, exist_ok=True)

METRICS = ["cosine_flat","kl_ab","js_divergence","mae"]
VALID_TEMPS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

def sheet_to_temp(name: str):
    s = name.strip().lower()
    
    if s.startswith("temp_") and s.count("_") >= 2:
        try:
            parts = s.split("_")
            if len(parts) >= 3:
                integer_part = int(parts[1])
                decimal_part = int(parts[2])
                temp = integer_part + decimal_part / 10.0
                return round(temp, 1)
        except:
            pass
    
    if "temp_" in s and "p" in s:
        try:
            temp_str = s.split("temp_")[1].replace("p", ".")
            return round(float(temp_str), 1)
        except:
            pass
    
    if "temp_" in s and "." in s:
        try:
            temp_str = s.split("temp_")[1]
            return round(float(temp_str), 1)
        except:
            pass
    
    return np.nan

def parse_all_excel_for(model_name: str, q: int):
    data_dir = os.path.join(BASE_DIR, f"pkl_outputs_{model_name}_q{q}")
    excel_paths = sorted(glob.glob(os.path.join(data_dir, f"metrics_q{q}_img*.xlsx")))

    if not excel_paths:
        print(f"     ⚠️  NO FILE: {data_dir}")
        return pd.DataFrame()

    rows = []
    for xp in excel_paths:
        basename = os.path.basename(xp)
        img_match = re.findall(r"img(\d+)\.xlsx", basename)
        img_id = int(img_match[0]) if img_match else basename
    
        try:
            xls = pd.ExcelFile(xp)
        except Exception as e:
            continue
        
        for sh in xls.sheet_names:
            try:
                df = pd.read_excel(xp, sheet_name=sh)
            except:
                continue
                
            if df is None or df.empty:
                continue

            temp_val = sheet_to_temp(sh)
            
            if pd.isna(temp_val) and "Temperature" in df.columns:
                temp_val = pd.to_numeric(df["Temperature"].iloc[0], errors='coerce')
                if not pd.isna(temp_val):
                    temp_val = round(temp_val, 1)
            
            if temp_val not in VALID_TEMPS:
                continue
            
            metric_means = {}
            for metric in METRICS:
                if metric in df.columns:
                    metric_means[metric] = df[metric].mean()
            
            if not metric_means:
                continue
            
            row = {
                "image_id": img_id,
                "model": model_name,
                "question": f"q{q}",
                "temperature": temp_val
            }
            row.update(metric_means)
            rows.append(row)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    result["image_id"] = result["image_id"].astype(str)
    
    return result

all_data = []

for m in MODELS:
    for q in QUESTIONS:
        dfq = parse_all_excel_for(m, q)
        if not dfq.empty:
            temps = sorted(dfq['temperature'].unique())
            unique_imgs = dfq['image_id'].nunique()
            all_data.append(dfq)
        else:
            print(f"   q{q}: NO DATA!")

if not all_data:
    raise ValueError("Could not read!")

all_data = pd.concat(all_data, ignore_index=True)

all_long = all_data.melt(
    id_vars=["model", "question", "image_id", "temperature"],
    value_vars=METRICS,
    var_name="metric",
    value_name="value"
).dropna(subset=["value"])

all_long_summary = (
    all_long
    .groupby(["metric", "model", "question", "temperature"], as_index=False)
    .agg(
        mean_value=("value", "mean"),
        std_value=("value", "std"),
        count=("value", "count")
    )
)

all_long_summary['std_value'] = all_long_summary['std_value'].fillna(0)

def normalize_per_model(df):
    df_norm = df.copy()
    df_norm['mean_normalized'] = 0.0
    df_norm['std_normalized'] = 0.0
    
    for metric_name in df['metric'].unique():
        for model_name in df['model'].unique():
            mask = (df_norm['metric'] == metric_name) & (df_norm['model'] == model_name)
            
            mean_vals = df_norm.loc[mask, 'mean_value']
            std_vals = df_norm.loc[mask, 'std_value']
            
            min_val = mean_vals.min()
            max_val = mean_vals.max()
            scale = max_val - min_val
            
            if scale > 1e-10:
                df_norm.loc[mask, 'mean_normalized'] = (mean_vals - min_val) / scale
                df_norm.loc[mask, 'std_normalized'] = std_vals / scale
            else:
                df_norm.loc[mask, 'mean_normalized'] = 0.5
                df_norm.loc[mask, 'std_normalized'] = 0.0
    
    return df_norm

all_long_summary = normalize_per_model(all_long_summary)

excel_path = os.path.join(OUT_DIR, "combined_normalized_data_with_std.xlsx")
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    all_long_summary.to_excel(writer, sheet_name='All_Data', index=False)
    
    for metric in sorted(all_long_summary['metric'].unique()):
        metric_data = all_long_summary[all_long_summary['metric'] == metric]
        sheet_name = metric.replace('_', ' ').title()[:31]
        metric_data.to_excel(writer, sheet_name=sheet_name, index=False)

sns.set_style("whitegrid")
questions_sorted = sorted(all_long_summary["question"].unique())
palette = sns.color_palette("husl", n_colors=len(questions_sorted))

for metric in sorted(all_long_summary["metric"].unique()):
    d = all_long_summary[all_long_summary["metric"] == metric].copy()
    if d.empty:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle(f"{metric} (Normalized) vs Temperature", fontsize=16, fontweight='bold')

    for idx, model in enumerate(MODELS):
        ax = axes[idx]
        model_data = d[d["model"] == model].copy()
        
        if model_data.empty:
            ax.set_title(f"{model}\n(No Data)")
            continue

        for q_idx, question in enumerate(questions_sorted):
            q_data = model_data[model_data["question"] == question].sort_values("temperature")
            
            if q_data.empty:
                continue
            
            ax.plot(
                q_data["temperature"], 
                q_data["mean_normalized"],
                marker='o',
                label=question,
                color=palette[q_idx],
                linewidth=2.5,
                markersize=7
            )
            
            ax.fill_between(
                q_data["temperature"],
                np.maximum(q_data["mean_normalized"] - q_data["std_normalized"], 0),
                np.minimum(q_data["mean_normalized"] + q_data["std_normalized"], 1),
                alpha=0.15,
                color=palette[q_idx]
            )

        ax.set_title(f"{model}", fontweight='bold', fontsize=14)
        ax.set_xlabel("Temperature", fontsize=12)
        if idx == 0:
            ax.set_ylabel(f"{metric} (normalized)", fontsize=12)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlim(-0.05, 1.05)
        ax.legend(title="Question", loc="best", framealpha=0.9)
        ax.grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f"combined_normalized_{metric}_subplots.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()

print(f"\nCompleted: {OUT_DIR}")


Completed: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed


In [ ]:
# === KALITELI GÖRSELLEŞTİRME === ONLY -- subplotlu
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

BASE_DIR = r"D:\HPdata\exp1_pkl_files"
OUT_DIR  = os.path.join(BASE_DIR, "plots_combined_normalized_with_std_fixed")
EXCEL_PATH = os.path.join(OUT_DIR, "combined_normalized_data_with_std.xlsx")

# Makale kalitesi için stil ayarları
plt.style.use('seaborn-v0_8-paper')
rcParams['font.family'] = 'serif'
rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']
rcParams['font.size'] = 11
rcParams['axes.labelsize'] = 12
rcParams['axes.titlesize'] = 13
rcParams['xtick.labelsize'] = 10
rcParams['ytick.labelsize'] = 10
rcParams['legend.fontsize'] = 10
rcParams['figure.titlesize'] = 14
rcParams['figure.dpi'] = 300
rcParams['savefig.dpi'] = 300
rcParams['savefig.bbox'] = 'tight'
rcParams['axes.linewidth'] = 1.2
rcParams['grid.linewidth'] = 0.8
rcParams['lines.linewidth'] = 2.0
rcParams['lines.markersize'] = 6

# Veriyi oku
print(f"📖 Veri okunuyor: {EXCEL_PATH}")
all_long_summary = pd.read_excel(EXCEL_PATH, sheet_name='All_Data')
print(f"✅ {len(all_long_summary)} satır okundu\n")

# Model ve soru sıralaması
MODELS = ["VILA8B", "LLAVAMed", "PRISM"]
questions_sorted = sorted(all_long_summary["question"].unique())

# Profesyonel renk paleti
colors_dict = {
    'q1': '#2E86AB',  # Mavi
    'q2': '#A23B72',  # Mor
    'q3': '#F18F01'   # Turuncu
}

# Metrik isimleri güzelleştirme
metric_labels = {
    'cosine_flat': 'Cosine Similarity',
    'kl_ab': 'KL Divergence',
    'js_divergence': 'JS Divergence',
    'mae': 'Mean Absolute Error'
}

print("📊 Grafikler oluşturuluyor...\n")

for metric in sorted(all_long_summary["metric"].unique()):
    d = all_long_summary[all_long_summary["metric"] == metric].copy()
    if d.empty:
        continue

    # 3 subplot oluştur
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    
    metric_title = metric_labels.get(metric, metric)
    fig.suptitle(f'{metric_title} vs Temperature', 
                 fontsize=14, fontweight='bold', y=1.02)

    for idx, model in enumerate(MODELS):
        ax = axes[idx]
        model_data = d[d["model"] == model].copy()
        
        if model_data.empty:
            ax.set_title(f"{model}\n(No Data)", fontsize=12)
            ax.set_xlabel("Temperature", fontsize=11)
            if idx == 0:
                ax.set_ylabel(f"Normalized {metric_title}", fontsize=11)
            continue

        for question in questions_sorted:
            q_data = model_data[model_data["question"] == question].sort_values("temperature")
            
            if q_data.empty:
                continue
            
            color = colors_dict.get(question, '#333333')
            
            # Ana çizgi
            ax.plot(
                q_data["temperature"], 
                q_data["mean_normalized"],
                marker='o',
                label=question.upper(),
                color=color,
                linewidth=2.0,
                markersize=6,
                markeredgewidth=0.5,
                markeredgecolor='white',
                alpha=0.9
            )
            
            # Standart sapma gölgeleme
            ax.fill_between(
                q_data["temperature"],
                np.maximum(q_data["mean_normalized"] - q_data["std_normalized"], 0),
                np.minimum(q_data["mean_normalized"] + q_data["std_normalized"], 1),
                alpha=0.15,
                color=color,
                linewidth=0
            )

        # Eksen ayarları
        ax.set_title(f"{model}", fontweight='bold', fontsize=12, pad=10)
        ax.set_xlabel("Temperature", fontsize=11)
        if idx == 0:
            ax.set_ylabel(f"Normalized {metric_title}", fontsize=11)
        
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlim(-0.05, 1.05)
        
        # Grid
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)
        
        # Kenar çizgileri
        for spine in ax.spines.values():
            spine.set_linewidth(1.2)
        
        # Legend - sadece ilk subplot'ta
        if idx == 0:
            ax.legend(title="Question", 
                     loc="best", 
                     framealpha=0.95,
                     edgecolor='gray',
                     fancybox=False,
                     shadow=False)

    plt.tight_layout()
    
    # Kaydet
    out_path = os.path.join(OUT_DIR, f"publication_{metric}.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    
    # PDF olarak da kaydet 
    pdf_path = os.path.join(OUT_DIR, f"publication_{metric}.pdf")
    plt.savefig(pdf_path, format='pdf', bbox_inches='tight', facecolor='white')
    
    plt.close()
    print(f"  ✓ {metric_title}")
    print(f"    → PNG: {out_path}")
    print(f"    → PDF: {pdf_path}")

print(f"\n🎉 Tüm grafikler oluşturuldu!")
print(f"📁 Klasör: {OUT_DIR}")

📖 Veri okunuyor: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\combined_normalized_data_with_std.xlsx
✅ 396 satır okundu

📊 Grafikler oluşturuluyor...

  ✓ Cosine Similarity
    → PNG: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_cosine_flat.png
    → PDF: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_cosine_flat.pdf
  ✓ JS Divergence
    → PNG: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_js_divergence.png
    → PDF: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_js_divergence.pdf
  ✓ KL Divergence
    → PNG: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_kl_ab.png
    → PDF: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_kl_ab.pdf
  ✓ Mean Absolute Error
    → PNG: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\publication_mae.png
    → PDF: D:\HPdata\exp1_p

In [5]:
# === Visualize data ===

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.ticker import MultipleLocator
import matplotlib.colors as mcolors


BASE_DIR = r"D:\HPdata\exp1_pkl_files"
OUT_DIR  = os.path.join(BASE_DIR, "plots_combined_normalized_with_std_fixed")
EXCEL_PATH = os.path.join(OUT_DIR, "combined_normalized_data_with_std.xlsx")


plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
rcParams['font.size'] = 12
rcParams['axes.labelsize'] = 14
rcParams['axes.titlesize'] = 15
rcParams['xtick.labelsize'] = 12
rcParams['ytick.labelsize'] = 12
rcParams['legend.fontsize'] = 11
rcParams['figure.dpi'] = 600
rcParams['savefig.dpi'] = 600
rcParams['savefig.bbox'] = 'tight'
rcParams['axes.linewidth'] = 1.5
rcParams['grid.linewidth'] = 1.0
rcParams['lines.linewidth'] = 2.5
rcParams['lines.markersize'] = 8


print(f"Data is read from: {EXCEL_PATH}")
all_long_summary = pd.read_excel(EXCEL_PATH, sheet_name='All_Data')
print(f"{len(all_long_summary)} lines were read\n")

MODELS = ["VILA8B", "LLAVAMed", "PRISM"]
MODEL_LABELS = {
    "VILA8B":   "VILA-M3-8B",
    "LLAVAMed": "LLaVA-Med v1.5",
    "PRISM":    "PRISM",
}
questions_sorted = sorted(all_long_summary["question"].unique())

colors_dict = {
    'q1': "#C04F8B",  
    'q2': "#db7e32",
    'q3': "#2E86AB",
}

metric_labels = {
    'cosine_flat': 'CS',#'Cosine Similarity',
    'kl_ab': 'KL Divergence',
    'js_divergence': 'JS Divergence',
    'mae': 'MAE'#'Mean Absolute Error'
}

def lighten_color(color, amount=0.5):

    try:
        c = mcolors.to_rgb(color)
        c_light = [c[i] + (1 - c[i]) * amount for i in range(3)]
        return mcolors.to_hex(c_light)
    except:
        return color

for metric in sorted(all_long_summary["metric"].unique()):
    d = all_long_summary[all_long_summary["metric"] == metric].copy()
    if d.empty:
        continue
    
    metric_title = metric_labels.get(metric, metric)
    
    
    for model in MODELS:
        model_data = d[d["model"] == model].copy()
        
        if model_data.empty:
            print(f"  ⚠️  {model} - {metric_title}: No data")
            continue
        
        fig, ax = plt.subplots(figsize=(5, 4))

        
        for question in questions_sorted:
            q_data = model_data[model_data["question"] == question].sort_values("temperature")
            
            if q_data.empty:
                continue

            main_color = colors_dict.get(question, '#333333')

            light_color = lighten_color(main_color, amount=0.6)

            ax.fill_between(
                q_data["temperature"],
                np.maximum(q_data["mean_normalized"] - q_data["std_normalized"], 0),
                np.minimum(q_data["mean_normalized"] + q_data["std_normalized"], 1),
                alpha=0.35,
                color=light_color,
                linewidth=0
            )

            ax.plot(
                q_data["temperature"], 
                q_data["mean_normalized"],
                marker='o',
                label=question.upper(),
                color=main_color,
                linewidth=2.5,
                markersize=8,
                markeredgewidth=1.5,
                markeredgecolor='white',
                markerfacecolor=main_color,
                alpha=1.0,
                zorder=10
            )
        
        if model == "VILA8B":
            model_lbl = "VILA-M3"
        elif model == "LLAVAMed":
            model_lbl = "LLaVA-Med"
        elif model == "PRISM":
            model_lbl = "PRISM"

        ax.set_ylabel(f"{metric_title} - {MODEL_LABELS[model]}", fontsize=14, fontweight='600')
        ax.set_xlabel("Temperature", fontsize=14, fontweight='600')

        ax.set_ylim(-0.05, 1.05)
        ax.set_xlim(-0.05, 1.05)

        ax.xaxis.set_major_locator(MultipleLocator(0.2))
        ax.yaxis.set_major_locator(MultipleLocator(0.2))

        ax.grid(True, which='major', alpha=0.6, linestyle='-', linewidth=1.0, color='#666666')
        ax.set_axisbelow(True)

        for spine in ax.spines.values():
            spine.set_linewidth(1.5)
            spine.set_color('#333333')

        ax.tick_params(axis='both', which='major', labelsize=12, 
                      length=6, width=1.5, direction='in', color='#333333')

        legend = ax.legend(
            title="Question", 
            loc="best", 
            framealpha=0.95,
            edgecolor='#333333',
            fancybox=False,
            shadow=False,
            frameon=True,
            fontsize=11,
            title_fontsize=11,
            borderpad=0.7,
            labelspacing=0.5,
            handlelength=2.5,
            handleheight=1.2
        )
        legend.get_frame().set_linewidth(1.5)
        legend.get_title().set_fontweight('600')
        
        plt.tight_layout()

        base_name = f"{metric}_{model}"
        
        png_path = os.path.join(OUT_DIR, f"{base_name}.png")
        plt.savefig(png_path, dpi=300, bbox_inches='tight', 
                    facecolor='white', edgecolor='none')
        
        pdf_path = os.path.join(OUT_DIR, f"{base_name}.pdf")
        plt.savefig(pdf_path, format='pdf', bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        
        plt.close()
        
        print(f"{os.path.basename(pdf_path)}")

print(f"\nAll plots are ready!")
print(f"📁 {OUT_DIR}\n")

Data is read from: D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed\combined_normalized_data_with_std.xlsx
396 lines were read

cosine_flat_VILA8B.pdf
cosine_flat_LLAVAMed.pdf
cosine_flat_PRISM.pdf
js_divergence_VILA8B.pdf
js_divergence_LLAVAMed.pdf
js_divergence_PRISM.pdf
kl_ab_VILA8B.pdf
kl_ab_LLAVAMed.pdf
kl_ab_PRISM.pdf
mae_VILA8B.pdf
mae_LLAVAMed.pdf
mae_PRISM.pdf

All plots are ready!
📁 D:\HPdata\exp1_pkl_files\plots_combined_normalized_with_std_fixed

